# Human-in-the-Loop Approval Agent with LangGraph

## Overview

Agents become risky when a model can move money, send messages, or mutate records without a deliberate checkpoint. This tutorial builds a small approval agent that classifies a proposed tool call, executes low-risk reads automatically, and pauses high-risk writes before any side effect. A reviewer can approve, reject, or replace the arguments; every transition is recorded in an audit log.

The example uses deterministic policies and simulated tools so it is safe to run without an API key. In a production agent, an LLM or planner would propose the same structured `action`, while the policy and approval boundary remain deterministic.

## Detailed Explanation

### Why approval belongs before the tool

A chat confirmation after a tool call is only a notification. A real approval gate must checkpoint the proposed action, stop execution, accept a reviewer decision, validate any edited arguments, and resume from the same state. Restart recovery additionally requires a persistent checkpointer. The policy below deliberately separates *planning* from *authorization*: a model may suggest a refund, but code decides whether a human must authorize it.

### Agent Architecture

![Human-in-the-Loop Approval Agent](../images/human-in-the-loop-approval-agent.svg)

The safe boundary is the edge between the approval gate and tool execution. Rejection never crosses that boundary; modification crosses it only after the replacement arguments pass the same validation as the original proposal.

## Required Packages

### Install LangGraph

The policy core uses only the Python standard library. This notebook pins LangGraph 0.2.76 for its interrupt/resume API; the repository-wide requirements retain an older version for existing tutorials. Run the notebook in its own environment. Focused graph tests explicitly skip unless this exact notebook version is installed.

In [ ]:
!pip install -q "langgraph==0.2.76"

## Implementation

### Imports and decision schema

The reviewer response is a constrained value rather than free-form prose. `modify` requires replacement arguments, while `approve` and `reject` preserve the original action.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timezone
from math import isfinite
from typing import Any, Dict, Optional


@dataclass(frozen=True)
class ApprovalDecision:
    """A reviewer's constrained response to one paused action."""
    decision: str
    reviewer: str
    reason: str = ""
    modified_args: Optional[Dict[str, Any]] = None

    def __post_init__(self) -> None:
        """Validate the decision vocabulary and modification payload."""
        if self.decision not in {"approve", "reject", "modify"}:
            raise ValueError("decision must be approve, reject, or modify")
        if self.decision == "modify" and self.modified_args is None:
            raise ValueError("modify requires modified_args")

### Define deterministic tool policies

Only explicitly allowlisted reads are low risk. Every refund is a mutating action and requires approval, regardless of amount. Validation is code, not an LLM judgment, so invalid amounts cannot be smuggled through a reviewer edit.

In [ ]:
def validate_action(action: Dict[str, Any]) -> None:
    """Reject unsupported tools and malformed tool arguments."""
    tool = action.get("tool")
    args = action.get("args", {})
    if not isinstance(args, dict):
        raise ValueError("action args must be a dictionary")
    if tool == "lookup_order":
        unexpected = set(args) - {"order_id"}
        if unexpected:
            raise ValueError(f"lookup_order received unexpected arguments: {sorted(unexpected)}")
        if not args.get("order_id"):
            raise ValueError("lookup_order requires order_id")
        return
    if tool == "issue_refund":
        unexpected = set(args) - {"order_id", "amount"}
        if unexpected:
            raise ValueError(f"issue_refund received unexpected arguments: {sorted(unexpected)}")
        if not args.get("order_id"):
            raise ValueError("issue_refund requires order_id")
        amount = args.get("amount")
        if (
            isinstance(amount, bool)
            or not isinstance(amount, (int, float))
            or not isfinite(amount)
            or amount <= 0
        ):
            raise ValueError("amount must be a finite positive number")
        return
    raise ValueError(f"unsupported tool: {tool}")


def classify_risk(action: Dict[str, Any]) -> str:
    """Allowlist reads as low risk and gate every mutating tool."""
    validate_action(action)
    if action["tool"] == "lookup_order":
        return "low"
    return "high"

### Create auditable request state

Each request starts with a structured action and an append-only list of events. Timestamps are created by the workflow, while tests and downstream systems can assert on event names and reviewer identity.

In [ ]:
def audit(event: str, **details: Any) -> Dict[str, Any]:
    """Create one timestamped audit event."""
    return {
        "event": event,
        "at": datetime.now(timezone.utc).isoformat(),
        **details,
    }


def new_request(tool: str, args: Dict[str, Any]) -> Dict[str, Any]:
    """Validate and initialize state for a proposed tool call."""
    action = {"tool": tool, "args": deepcopy(args)}
    validate_action(action)
    return {
        "action": action,
        "status": "planned",
        "approval_request": None,
        "audit_log": [audit("action_proposed", action=deepcopy(action))],
    }

### Simulate tools and isolate the side-effect boundary

The tools return deterministic payloads instead of touching a real order system. Crucially, `execute_tool` is the only dispatcher and independently refuses high-risk actions unless the state was authorized by a reviewer decision. This makes the approval invariant hold even when a caller bypasses the normal workflow.

In [ ]:
def lookup_order(order_id: str) -> Dict[str, Any]:
    """Return deterministic order data for the safe tutorial."""
    return {"order_id": order_id, "status": "paid", "total": 240}


def issue_refund(order_id: str, amount: float) -> Dict[str, Any]:
    """Return a simulated refund receipt without moving money."""
    return {"order_id": order_id, "refunded": amount, "simulated": True}


TOOLS = {"lookup_order": lookup_order, "issue_refund": issue_refund}


def execute_tool(state: Dict[str, Any], reviewer: Optional[str] = None) -> Dict[str, Any]:
    """Execute a validated tool and append its auditable receipt."""
    updated = deepcopy(state)
    action = updated["action"]
    risk = classify_risk(action)
    if risk == "high" and updated.get("status") != "authorized":
        raise PermissionError("high-risk actions require authorization")
    updated["tool_result"] = TOOLS[action["tool"]](**action["args"])
    updated["status"] = "completed"
    updated["approval_request"] = None
    updated["audit_log"].append(
        audit("tool_executed", tool=action["tool"], reviewer=reviewer)
    )
    return updated

### Pause at the approval gate

Low-risk actions cross the boundary immediately. A high-risk action returns serializable approval context and stops before calling a tool.

In [ ]:
def run_until_pause(state: Dict[str, Any]) -> Dict[str, Any]:
    """Execute allowlisted reads or return a serializable approval request."""
    updated = deepcopy(state)
    risk = classify_risk(updated["action"])
    updated["audit_log"].append(audit("risk_classified", risk=risk))
    if risk == "low":
        return execute_tool(updated)

    updated["status"] = "awaiting_approval"
    updated["approval_request"] = {
        "risk": risk,
        "action": deepcopy(updated["action"]),
        "allowed_decisions": ["approve", "reject", "modify"],
    }
    updated["audit_log"].append(audit("approval_requested", risk=risk))
    return updated

### Resume with approve, reject, or modify

A rejected request terminates without a tool result. Approval executes the persisted proposal. Modification replaces only the arguments, revalidates them, and records both the decision and the reviewer before execution.

In [ ]:
def resume_with_decision(
    state: Dict[str, Any], decision: ApprovalDecision
) -> Dict[str, Any]:
    """Apply a review decision and execute only authorized actions."""
    if state.get("status") != "awaiting_approval":
        raise ValueError("request is not awaiting approval")

    updated = deepcopy(state)
    if decision.decision == "reject":
        updated["status"] = "rejected"
        updated["approval_request"] = None
        updated["audit_log"].append(
            audit("action_rejected", reviewer=decision.reviewer, reason=decision.reason)
        )
        return updated

    if decision.decision == "modify":
        before = deepcopy(updated["action"]["args"])
        candidate = deepcopy(updated["action"])
        candidate["args"] = deepcopy(decision.modified_args)
        validate_action(candidate)
        updated["action"] = candidate
        updated["audit_log"].append(
            audit(
                "action_modified",
                reviewer=decision.reviewer,
                before=before,
                after=deepcopy(candidate["args"]),
            )
        )

    updated["audit_log"].append(
        audit("action_authorized", reviewer=decision.reviewer, decision=decision.decision)
    )
    updated["status"] = "authorized"
    return execute_tool(updated, reviewer=decision.reviewer)

### Wrap the policy with LangGraph interrupt and resume

The policy node first returns `awaiting_approval`, so that state and its audit event are committed before the next node calls `interrupt`. The caller receives `__interrupt__`, obtains a reviewer decision, and resumes the same thread with `Command(resume=...)`. `InMemorySaver` supports this only while the Python process remains alive; restart recovery requires a persistent database-backed checkpointer.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class GraphState(TypedDict, total=False):
    """Serializable state persisted around the approval interrupt."""
    action: Dict[str, Any]
    status: str
    approval_request: Optional[Dict[str, Any]]
    audit_log: list
    tool_result: Dict[str, Any]


def policy_node(state: GraphState) -> GraphState:
    """Commit either a completed read or an awaiting-approval state."""
    return run_until_pause(dict(state))


def route_after_policy(state: GraphState) -> str:
    """Route only persisted high-risk requests to human review."""
    if state["status"] == "awaiting_approval":
        return "approval_gate"
    return END


def approval_node(state: GraphState) -> GraphState:
    """Interrupt a persisted request and apply its structured decision."""
    raw_decision = interrupt(state["approval_request"])
    return resume_with_decision(dict(state), ApprovalDecision(**raw_decision))


builder = StateGraph(GraphState)
builder.add_node("policy", policy_node)
builder.add_node("approval_gate", approval_node)
builder.add_edge(START, "policy")
builder.add_conditional_edges(
    "policy", route_after_policy, {"approval_gate": "approval_gate", END: END}
)
builder.add_edge("approval_gate", END)
approval_graph = builder.compile(checkpointer=InMemorySaver())

## Usage Example

### Low-risk reads complete immediately

No reviewer is needed for a read-only lookup.

In [ ]:
lookup = run_until_pause(new_request("lookup_order", {"order_id": "A-100"}))
print(lookup["status"], lookup["tool_result"])

### High-risk refunds pause and resume

The first invocation stops with approval context and no `tool_result`. The second resumes the same thread with a reviewer-approved lower amount.

In [ ]:
config = {"configurable": {"thread_id": "refund-demo-1"}}
request = new_request(
    "issue_refund", {"order_id": "A-100", "amount": 240}
)
updates = list(
    approval_graph.stream(request, config=config, stream_mode="updates")
)
approval_request = next(
    update["__interrupt__"][0].value
    for update in updates
    if "__interrupt__" in update
)
paused = approval_graph.get_state(config).values
print("Paused:", approval_request)

completed = approval_graph.invoke(
    Command(
        resume={
            "decision": "modify",
            "reviewer": "ops@example.com",
            "reason": "Approve within discretionary limit",
            "modified_args": {"order_id": "A-100", "amount": 80},
        }
    ),
    config=config,
)
print("Completed:", completed["tool_result"])

### Verify the safety invariants

These assertions are executable documentation: the graph must pause before a side effect and must use the reviewer's validated amount after resuming.

In [ ]:
assert paused["status"] == "awaiting_approval"
assert paused["approval_request"] == approval_request
assert paused["audit_log"][-1]["event"] == "approval_requested"
assert "tool_result" not in paused
assert completed["status"] == "completed"
assert completed["tool_result"]["refunded"] == 80
assert completed["tool_result"]["simulated"] is True
print("Safety invariants verified")

## Comparison

| Pattern | Stops before side effect | Restart recovery | Reviewer can edit arguments | Auditability |
|---|---:|---:|---:|---:|
| Ask for confirmation in the prompt | No guarantee | No | Unstructured | Low |
| Approve every tool call | Yes | Depends on runtime | Usually | High, but noisy |
| Risk-based interrupt (this tutorial) | Yes | Process lifetime here; persistent checkpointer required | Yes, then revalidated | High |
| Fully autonomous tools | No | Not applicable | No | Depends on tracing |

Risk-based approval preserves autonomy for harmless reads while reserving human attention for consequential actions. It is more code than a prompt-level confirmation because it provides a real authorization boundary rather than a conversational convention.

## Additional Considerations

- **Use durable checkpoints.** `InMemorySaver` is appropriate for a tutorial, not a service restart. Use a supported database checkpointer and encrypt sensitive state.
- **Bind identity server-side.** Do not trust a reviewer email supplied by the browser. Resolve the authenticated principal in the approval API and enforce role or amount limits there.
- **Make approvals expire.** Revalidate inventory, balances, policy versions, and target records when a stale approval resumes.
- **Protect against double execution.** Give mutation tools an idempotency key tied to the request, and persist the execution receipt.
- **Show the exact effect.** Approval UI should display tool name, normalized arguments, policy reason, and a diff when arguments were modified.
- **Keep policy deterministic.** An LLM can explain risk, but hard authorization limits should remain testable code.
- **Escalate unknown tools.** This tutorial rejects them. Production systems should default-deny missing policy entries rather than silently treating them as low risk.

## References

- [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [OWASP Agentic AI Threats and Mitigations](https://genai.owasp.org/resource/agentic-ai-threats-and-mitigations/)